# Item-set Lattice - Starter Notebook
Visualizes the item-set lattice (power-set DAG) showing support values and frequent/infrequent boundaries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations

In [ ]:
transactions = [
    {'A','B','C'},
    {'A','B'},
    {'A','C'},
    {'B','C'},
    {'A','B','C'},
    {'B','C'},
    {'A','C'},
    {'A','B','C'}
]
ITEMS = sorted({'A','B','C'})
N = len(transactions)
MIN_SUPPORT = 0.4

In [ ]:
def support(itemset):
    fs = frozenset(itemset)
    return sum(1 for t in transactions if fs <= t) / N

# Build all non-empty subsets
all_sets = []
for k in range(1, len(ITEMS)+1):
    for combo in combinations(ITEMS, k):
        fs = frozenset(combo)
        all_sets.append((fs, support(fs)))

for fs, sup in all_sets:
    tag = 'FREQUENT' if sup >= MIN_SUPPORT else 'infrequent'
    label = '{' + ', '.join(sorted(fs)) + '}'
    print(f'{label:<20}  support={sup:.2f}  [{tag}]')

In [ ]:
# Visualize the 3-item lattice
# Layout: level 0 = empty (not shown), level 1 = singletons, level 2 = pairs, level 3 = triple
node_pos = {
    frozenset(['A']):     (1, 2),
    frozenset(['B']):     (3, 2),
    frozenset(['C']):     (5, 2),
    frozenset(['A','B']): (1, 1),
    frozenset(['A','C']): (3, 1),
    frozenset(['B','C']): (5, 1),
    frozenset(['A','B','C']): (3, 0),
}
edges = [
    (frozenset(['A','B']),   frozenset(['A'])),
    (frozenset(['A','B']),   frozenset(['B'])),
    (frozenset(['A','C']),   frozenset(['A'])),
    (frozenset(['A','C']),   frozenset(['C'])),
    (frozenset(['B','C']),   frozenset(['B'])),
    (frozenset(['B','C']),   frozenset(['C'])),
    (frozenset(['A','B','C']), frozenset(['A','B'])),
    (frozenset(['A','B','C']), frozenset(['A','C'])),
    (frozenset(['A','B','C']), frozenset(['B','C'])),
]

sup_map = {fs: s for fs, s in all_sets}

fig, ax = plt.subplots(figsize=(8, 5))
for child, parent in edges:
    x0, y0 = node_pos[child]
    x1, y1 = node_pos[parent]
    ax.plot([x0, x1], [y0, y1], 'k-', lw=1, alpha=0.5)

for fs, (x, y) in node_pos.items():
    sup = sup_map.get(fs, 0)
    color = '#2ecc71' if sup >= MIN_SUPPORT else '#e74c3c'
    ax.scatter(x, y, s=800, c=color, zorder=3, edgecolors='k')
    label = ','.join(sorted(fs))
    ax.text(x, y+0.12, f'{{{label}}}\n{sup:.2f}', ha='center', fontsize=8)

ax.set_xlim(0, 6)
ax.set_ylim(-0.5, 2.6)
ax.set_yticks([0,1,2])
ax.set_yticklabels(['3-sets', '2-sets', '1-sets'])
ax.set_xticks([])
ax.set_title(f'Item-set Lattice (min_support={MIN_SUPPORT})')
freq_patch = mpatches.Patch(color='#2ecc71', label='Frequent')
infreq_patch = mpatches.Patch(color='#e74c3c', label='Infrequent')
ax.legend(handles=[freq_patch, infreq_patch])
plt.tight_layout()
plt.show()